# Configure & Run Experiment

Fill in the fields below to build a config and run the full 4-step
pipeline (preprocess -> run model -> postprocess -> plot). Advanced/rarely-
changed settings are collapsed under "Advanced settings".

**Before you start**: set **Experiment root** to a directory you own
(not the instructor's data) — you write your own runs there, and
**Preprocess dir** to wherever the base climatology `.pt` files you want to
use already live (e.g. one generated for you, or one you built in the
other notebook here).

If **Cold start** is checked and the target experiment directory already
exists, you'll be asked to confirm before anything is deleted.


In [ ]:
import sys, os

def _find_project_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.exists(os.path.join(d, "scripts", "_config.py")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("Could not find the project root (looked for scripts/_config.py above "
                        + start + "). Make sure this notebook is somewhere inside the repo.")

PROJECT_ROOT = _find_project_root(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts"))
print("Project root:", PROJECT_ROOT)


In [ ]:
import subprocess
import ipywidgets as w
import yaml

# --- Curated (main) fields ---
r_model_type = w.Dropdown(options=["fixed_season", "gamma_ac"], value="fixed_season", description="Model:")
r_season = w.Dropdown(options=["DJF", "JJA", "MAM", "SON"], value="DJF", description="Season:")
r_heating_name = w.Text(description="Heating name:")
r_heating_source = w.Dropdown(options=["custom", "cca", "cesm2", "era5"], value="custom", description="Heating source:")
r_heating_file = w.Text(description="Heating file:", placeholder="(source=custom)")
r_start_year = w.IntText(value=1999, description="Start yr:")
r_end_year = w.IntText(value=2020, description="End yr:")
r_preprocess_path = w.Text(description="Preprocess dir:", placeholder="path to base climatology .pt files")
r_experiment_root = w.Text(description="Experiment root:", placeholder="YOUR OWN output directory")
r_experiment_name = w.Text(description="Experiment name:", placeholder="directory name under experiment_root")
r_run_length_days = w.IntText(value=30, description="Run length (days):")
r_cold_start = w.Checkbox(value=True, description="Cold start")
r_toffset = w.IntText(value=0, description="Restart offset (days):")
r_shape_file = w.Text(description="Shape file:", placeholder="(gamma_ac) e.g. shapeAC.pt")
r_scale_file = w.Text(description="Scale file:", placeholder="(gamma_ac) e.g. scaleAC.pt")
r_control_experiment = w.Text(description="Control exp:", placeholder="(optional) name of a control run to diff against")
r_spinup_days = w.IntText(value=60, description="Spinup days:")
r_plot_vars = w.SelectMultiple(options=["uvel", "vvel", "geo", "temp"], value=["uvel", "vvel", "geo"], description="Plot vars:")

# --- Advanced fields (collapsed) ---
a_zw = w.Dropdown(options=[42, 63, 124], value=63, description="zw:")
a_kmax = w.Dropdown(options=[11, 26], value=26, description="kmax:")
a_chunk_size_days = w.IntText(value=30, description="Chunk size (days):")
a_compute_slp = w.Checkbox(value=False, description="Compute SLP")
a_postprocess_vars = w.SelectMultiple(options=["uvel", "vvel", "geo", "temp"], value=["uvel", "vvel", "geo"], description="Postprocess vars:")

advanced_box = w.Accordion(children=[w.VBox([a_zw, a_kmax, a_chunk_size_days, a_compute_slp, a_postprocess_vars])])
advanced_box.set_title(0, "Advanced settings (usually leave as default)")
advanced_box.selected_index = None  # collapsed by default

config_output = w.Output()
config_path_box = w.Text(description="Save config to:", placeholder="e.g. my_experiment.yaml")
build_config_button = w.Button(description="Build Config", button_style="info")

run_output = w.Output()
run_button = w.Button(description="Run Pipeline (steps 1-4)", button_style="primary")
confirm_box = w.VBox([])  # populated with a confirmation button when needed


def on_model_type_change(change):
    is_fixed = change["new"] == "fixed_season"
    r_season.layout.display = "" if is_fixed else "none"
    r_heating_source.options = (["custom", "cca", "cesm2", "era5"] if is_fixed
                                 else ["custom", "default_enso_composite"])
    r_heating_source.value = r_heating_source.options[0]
    r_shape_file.layout.display = "none" if is_fixed else ""
    r_scale_file.layout.display = "none" if is_fixed else ""


r_model_type.observe(on_model_type_change, names="value")
on_model_type_change({"new": r_model_type.value})


def build_cfg_dict():
    if not r_heating_name.value:
        raise ValueError("Heating name is required.")
    if not r_preprocess_path.value:
        raise ValueError("Preprocess dir is required.")
    if not r_experiment_root.value:
        raise ValueError("Experiment root is required (use your own directory).")
    if not r_experiment_name.value:
        raise ValueError("Experiment name is required.")

    cfg = {
        "model_type": r_model_type.value,
        "start_year": r_start_year.value,
        "end_year": r_end_year.value,
        "season": r_season.value if r_model_type.value == "fixed_season" else "annual",
        "heating_name": r_heating_name.value,
        "heating_source": r_heating_source.value,
        "preprocess_path_override": r_preprocess_path.value,
        "experiment_root": r_experiment_root.value,
        "experiment_name": r_experiment_name.value,
        "run_length_days": r_run_length_days.value,
        "cold_start": r_cold_start.value,
        "toffset": r_toffset.value,
        "control_experiment": r_control_experiment.value or None,
        "spinup_days": r_spinup_days.value,
        "plot_vars": list(r_plot_vars.value),
        "zw": a_zw.value,
        "kmax": a_kmax.value,
        "chunk_size_days": a_chunk_size_days.value,
        "compute_slp": a_compute_slp.value,
        "postprocess_vars": list(a_postprocess_vars.value),
    }
    if r_model_type.value == "fixed_season":
        cfg["model_subtype"] = "weakly_prescribed_mean"
        if r_heating_file.value:
            cfg["heating_file_override"] = r_heating_file.value
    else:
        if r_shape_file.value:
            cfg["shape_file_override"] = r_shape_file.value
        if r_scale_file.value:
            cfg["scale_file_override"] = r_scale_file.value
    return cfg


def on_build_config_clicked(b):
    with config_output:
        config_output.clear_output()
        try:
            cfg = build_cfg_dict()
            path = config_path_box.value or "student_experiment.yaml"
            with open(path, "w") as f:
                yaml.safe_dump(cfg, f, sort_keys=False)
            print(f"Wrote config to {path}")
            print(yaml.safe_dump(cfg, sort_keys=False))
        except Exception as e:
            print(f"ERROR: {e}")


build_config_button.on_click(on_build_config_clicked)


def experiment_dir():
    return os.path.join(r_experiment_root.value, r_experiment_name.value)


def _run_pipeline():
    with run_output:
        run_output.clear_output()
        config_path = config_path_box.value or "student_experiment.yaml"
        if not os.path.exists(config_path):
            print("ERROR: build the config first.")
            return
        steps = ["01_preprocess.py", "02_run_model.py", "03_postprocess.py", "04_plot_results.py"]
        for step in steps:
            cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", step),
                   "--config", os.path.abspath(config_path)]
            print(f"--- Running {step} ---")
            result = subprocess.run(cmd, cwd=os.path.join(PROJECT_ROOT, "scripts"),
                                     capture_output=True, text=True)
            print(result.stdout)
            if result.returncode != 0:
                print(result.stderr)
                print(f"FAILED at {step} (exit {result.returncode}) — stopping.")
                return
        print("Pipeline complete.")


def on_run_clicked(b):
    with run_output:
        run_output.clear_output()
    target = experiment_dir()
    if r_cold_start.value and os.path.isdir(target):
        confirm_btn = w.Button(description=f"Confirm: delete and restart {target}", button_style="danger")

        def on_confirm(cb):
            confirm_box.children = ()
            _run_pipeline()

        confirm_btn.on_click(on_confirm)
        confirm_box.children = (confirm_btn,)
        with run_output:
            print(f"cold_start=True and {target} already exists — click above to confirm overwrite.")
    else:
        _run_pipeline()


run_button.on_click(on_run_clicked)

run_panel = w.VBox([
    w.HTML("<b>Configure &amp; Run Experiment</b>"),
    r_model_type, r_season, r_heating_name, r_heating_source, r_heating_file,
    r_start_year, r_end_year, r_preprocess_path,
    r_experiment_root, r_experiment_name, r_run_length_days,
    r_cold_start, r_toffset, r_shape_file, r_scale_file,
    r_control_experiment, r_spinup_days, r_plot_vars,
    advanced_box,
    config_path_box, build_config_button, config_output,
    run_button, confirm_box, run_output,
])

display(run_panel)
